# PC-FNO3D réel (Grand Tunis) — validation sur de VRAIES mesures (Colab, GPU)

Sur les 180 instants de test (2022) : entrée de production avec les **vrais** anémomètres
(METAR Tunis-Carthage, SYNOP Zaghouan et Nabeul), chaque station **retirée de l'entrée** quand
on la juge ; vent à 10 m contre la mesure, comparé à ERA5 et au générateur ; profils contre les
**radiosondages** de Tunis-Carthage (0–1 000 m). Modèles : v2 prolongé, puis v2.
`scripts/valider_grand_tunis_reel.py`. La session s'arrête à la fin.

Sur le Drive : `grand_tunis_obs.zip` (python scripts/package_grand_tunis_colab.py).
Résultats : `runs/<modèle>/validation_reelle/test/`.

## 1. Drive, code et données

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import json, os, shutil, subprocess, sys, time, zipfile
from pathlib import Path

GT = Path("/content/drive/MyDrive/pcfno3d_grand_tunis")
GT.mkdir(parents=True, exist_ok=True)
JOURNAUX = GT / "journaux"
JOURNAUX.mkdir(exist_ok=True)

# code : archive deposee sur le Drive depuis ines-pc
CODE = Path("/content/PFE")
if CODE.exists():
    shutil.rmtree(CODE)
CODE.mkdir(parents=True)
with zipfile.ZipFile(GT / "pcfno3d_code.zip") as zf:
    zf.extractall(CODE)
tampon = CODE / "PCFNO3D_VERSION.txt"
print(tampon.read_text().strip() if tampon.exists() else "tampon : absent")
%cd /content/PFE
!pip install -q --no-deps -e . 2>&1 | tail -2
!pip install -q pandera xarray netCDF4 h5netcdf 2>&1 | tail -2
sys.path[:0] = ["/content/PFE", "/content/PFE/src"]

def lancer(nom: str, commande: str) -> None:
    """Commande obligatoire ; sortie affichee ET journalisee sur le Drive ; arret si echec."""
    t0 = time.time()
    with open(JOURNAUX / f"{nom}.log", "a", encoding="utf-8") as log:
        log.write(f"\n===== {time.strftime('%Y-%m-%d %H:%M:%S')} : {commande}\n")
        p = subprocess.Popen(commande, shell=True, cwd="/content/PFE", stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT, text=True, bufsize=1)
        for ligne in p.stdout:
            print(ligne, end="")
            log.write(ligne)
            log.flush()
        p.wait()
    print(f"[{nom}] {time.time() - t0:.0f} s, code {p.returncode}")
    if p.returncode:
        raise RuntimeError(f"{nom} a echoue (code {p.returncode}) : voir {JOURNAUX / (nom + '.log')}")

# donnees : grille reelle, forcage ERA5, stations (une archive sur le Drive)
DON = Path("/content/donnees")
if not (DON / "grand_tunis" / "forcage_era5.csv").exists():
    with zipfile.ZipFile(GT / "grand_tunis_donnees.zip") as zf:
        zf.extractall(DON)
# mises a jour deposees apres coup sur le Drive (morceaux de moins de 10 Mo) :
# maj_<fichier> ou maj_<fichier>.000, .001... -> remplacent <fichier> dans donnees/grand_tunis
import collections
morceaux = collections.defaultdict(list)
for f in sorted(GT.glob("maj_*")):
    nom = f.name[4:]
    tete, _, queue = nom.rpartition(".")
    morceaux[tete if queue.isdigit() and tete else nom].append(f)
for nom, fs in morceaux.items():
    with open(DON / "grand_tunis" / nom, "wb") as out:
        for f in sorted(fs):
            out.write(f.read_bytes())
    print("mise a jour :", nom, "<-", [f.name for f in sorted(fs)])
print(sorted(p.name for p in (DON / "grand_tunis").iterdir()))

## 2. Instants de test et observations

In [ ]:
# instants de TEST des 3 parties (S, Y, E) + index fusionne ; observations reelles du domaine
LOC = Path("/content/grand_tunis")
CACHE, JEU = LOC / "cache", LOC / "jeu"
CACHE.mkdir(parents=True, exist_ok=True)
JEU.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(GT / "jeu.zip") as zf:
    zf.extractall(JEU)
import pandas as pd
index, n = [], 0
for k in (1, 2, 3):
    with zipfile.ZipFile(GT / f"cache_partie_{k}.zip") as zf:
        with zf.open(f"index_partie_{k}.csv") as f:
            idx = pd.read_csv(f)
        index.append(idx)
        test = set(idx.loc[idx["split"] == "test", "stem"])
        noms = [x for x in zf.namelist() if x.rsplit("_", 1)[0] in test]
        zf.extractall(CACHE, members=noms)
        n += len(noms)
pd.concat(index).drop_duplicates("stem").to_csv(CACHE / "index.csv", index=False)
OBS = Path("/content/obs")
with zipfile.ZipFile(GT / "grand_tunis_obs.zip") as zf:
    zf.extractall(OBS)
print(f"{n} fichiers de test ; observations : {sorted(p.name for p in OBS.iterdir())}")

## 3. Validation (v2 prolongé, puis v2), puis fin de la session

In [ ]:
from google.colab import runtime
try:
    for conf, run in (("grand_tunis_v2_long_colab", "grand_tunis_v2_long"),
                      ("grand_tunis_v2_colab", "grand_tunis_v2")):
        try:
            lancer(f"valider_{run}", f"python -u scripts/valider_grand_tunis_reel.py -c {conf} --obs {OBS}")
        except Exception as e:
            print(e)
finally:
    runtime.unassign()